## TUGAS MANDIRI 
---

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh **"Kopi Nusantara"**, jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data dan mengunggah data transaksi ke HDFS.

In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


### Instruksi Pengerjaan

Buat `SparkSession`

---

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9



**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

---

In [6]:
# Membaca ketiga sumber data dari HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

---

In [7]:
# Menggabungkan ketiga tabel
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

# Menambahkan kolom total_penjualan
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

---

In [13]:
# Mengelompokkan cabang beserta menunya
ringkasan = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Menerapkan window function untuk mengambil top-2 menu terlaris di setiap cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan.withColumn("top_menu", row_number().over(window_cabang)) \
                     .filter(col("top_menu") <= 2)

top2_menu.orderBy("nama_cabang", "top_menu").show()

+--------------------+---------+--------------+---------------+--------+
|         nama_cabang|nama_menu| kategori_menu|total_penjualan|top_menu|
+--------------------+---------+--------------+---------------+--------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|        8505000|       1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|        8085000|       2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|        8750000|       1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|        8715000|       2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|        9275000|       1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|        8160000|       2|
+--------------------+---------+--------------+---------------+--------+



**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

---

In [14]:
# Mendaftarkan df_gabungan sebagai tabel sementara
df_gabungan.createOrReplaceTempView("gabungan")

# Menampilkan total penjualan per kepala_barista dari yang tertinggi
penjualan = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
penjualan.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

---

In [21]:
# Menyimpan data gabungan ke HDFS sebagai Parquet
!hdfs dfs -mkdir -p /user/jauharaputri/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/jauharaputri/tugas7/processed/laporan"
)

# Verifikasi
df_laporan = spark.read.parquet("hdfs://localhost:9000/user/jauharaputri/tugas7/processed/laporan")
print("Data gabungan:", df_laporan.count(), "baris")
df_laporan.show(5)

Data gabungan: 4000 baris
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|        3|  KN15|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-12|35000|      

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---

In [20]:
# Membuat tabel ringkasan
ringkasan_eksekutif = df_laporan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



**Penjelasan:** Berdasarkan tabel ringkasan, cabang yang berkinerja terbaik adalah cabang Semarang dengan total penjualan sebesar Rp.90.542.000 dari 1.393 transaksi. Cabang Magelang berada di posisi kedua setelah cabang Semarang dengan total Rp87.138.000 dari 1.302 transaksi. Walaupun jumlah transaksinya paling sedikit, namun rata-rata nilai transaksinya justru paling tinggi di antara ketiga cabang. Cabang Yogyakarta berada di posisi terakhir dengan kinerja terlemah mendapat total penjualan Rp.84.645.000 dari 1.305 transaksi dan rata-rata nilai transaksinya pun paling rendah. Berdasarkan top-2 menu terlaris, Menu-12 berkategori makanan ringan menjadi yang terlaris di Yogyakarta dengan total penjualan sebesar Rp9.275.000 dan pada posisi kedua yaitu Menu-15 berkategori non-kopi dengan total Rp.8.160.000. Oleh karenanya, menu yang sebaiknya dipromosikan lebih gencar di cabang ini adalah Menu-12 yang menjadi top 1 yang juga diminati pada cabang Magelang karena masuk top-2 menu terlaris. Menu-12 ini dapat dipromosikan dengan Menu-15 yang menjadi top-2 cabang Yogyakarta sebagai pendamping dari makanan ringan itu sendiri, paket bundling dengan minuman.

---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [ ]:
spark.stop()
print("SparkSession ditutup.")